### 1. Import Library (PyTorch & Transformers)
Dalam pemrosesan *Large Language Models* (LLM) atau NLP modern, ekosistem **Hugging Face Transformers** hampir selalu digabungkan dengan **PyTorch** sebagai backend komputasinya.

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from transformers import BertTokenizer, BertForSequenceClassification
import pandas as pd
import numpy as np

### 2. Pengecekan Perangkat (Device: CPU vs GPU)
Model NLP berukuran besar (ratusan juta hingga miliaran parameter) wajib dijalankan di atas GPU (CUDA) apabila tersedia agar proses *training* dan *inference* tidak memakan waktu berhari-hari.

In [2]:
device = torch.device(torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu')
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device yang digunakan: {device}")

Device yang digunakan: cuda


### 3. Dasar Tensor PyTorch (Batu Bata Utama Large Model)
Sebelum kita melatih LLM, pahami bahwa model hanya melihat array multi-dimensi (Tensor). Kemampuan merubah bentuk (`view`/`reshape`) dan tipe data sangat krusial saat menyesuaikan dimensi input ke dalam model LLM.

In [3]:
x = torch.tensor([1,2,3,4,5,6], dtype=torch.float32)

x_reshaped = x.view(2,3)
x_gpu = x_reshaped.to(device)

x_gpu.device

device(type='cuda', index=0)

### 4. Penggunaan `pipeline` Hugging Face (Bonus Trik Cepat)
Jika di kompetisi Anda diminta untuk '*cukup lakukan inferensi sentimen dengan model yang sudah ada (zero-shot) tanpa menulis kodingan rumit*', Anda wajib memanfaatkan abstraksi paling tingkat tinggi di Transformers: yaitu **Pipeline**.

In [4]:
from transformers import pipeline

nlp_analyzer = pipeline('sentiment-analysis')

text = " am very excited to join this amazing hackathon!"
result = nlp_analyzer(text)

result

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9997619986534119}]

### 5. Inisialisasi Tokenizer & Konversi Teks
`Tokenizer` berfungsi memecah teks berupa kalimat menjadi potongan kata/sub-kata (token), lalu mengubahnya menjadi angka (`input_ids`) yang bisa dipahami komputer.

*Parameter penting:*
- `padding='max_length'`: Menambah angka 0 (PAD) jika kalimat terlalu pendek.
- `truncation=True`: Memotong kalimat jika melebihi `max_length`.
- `return_tensors='pt'`: Mengembalikan output dalam bentuk objek PyTorch Tensor.

In [14]:
# Model merujuk pada nama model di HuggingFace hub atau path folder lokal

bert_path = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(bert_path)

sample_text = ["Huawei global competition is highly competitive."]
inputs = tokenizer(
    sample_text, 
    max_length=12, 
    padding='max_length', 
    truncation=True, 
    return_tensors='pt'
)

print("Input IDs (Angka representasi token):\n", inputs['input_ids'])
print("\nAttention Mask (1=kata asli, 0=padding):\n", inputs['attention_mask'])
print("\ntoken type", inputs['token_type_ids'])

Input IDs (Angka representasi token):
 tensor([[  101, 23064, 19845,  3795,  2971,  2003,  3811,  6975,  1012,   102,
             0,     0]])

Attention Mask (1=kata asli, 0=padding):
 tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]])

token type tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])


### 6. Pemahaman Tokenizer Lanjutan (Decode & Special Tokens)
Model NLP seperti BERT memiliki kamus vocab sendiri dan menambahkan token spesial secara otomatis. `[CLS]` (token awal untuk klasifikasi) dan `[SEP]` (token pemisah kalimat). Mari kita "bedah" isi dari angka-angka ID tersebut.

In [67]:
kalimat = ["Huawei's large models are very fast!", "Indonesia mantap tenang"]
input_ids = tokenizer.encode(kalimat, add_special_tokens=True, max_length=12, padding='max_length', truncation=True)
print(f"id : {input_ids}")

tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
print(f"token : {tokens}")

reconstruct = tokenizer.decode(input_ids)
print(f"reconstruct : {reconstruct}")

id : [[101, 30522, 1005, 1055, 2312, 4275, 2024, 2200, 3435, 999, 102, 0], [101, 6239, 2158, 2696, 2361, 2702, 5654, 102, 0, 0, 0, 0]]
token : ['[CLS]', 'huawei', "'", 's', 'large', 'models', 'are', 'very', 'fast', '!', '[SEP]', '[PAD]']
reconstruct : ["[CLS] huawei ' s large models are very fast! [SEP] [PAD]", '[CLS] indonesia mantap tenang [SEP] [PAD] [PAD] [PAD] [PAD]']


### 7. Menambahkan Kosakata & Memodifikasi Special Token
Saat merekayasa data, kadang istilah spesifik dari ranah kompetisi/perusahaan (misal: "Huawei", "MindSpore") atau Special Token kustom (misal: penanda awal prompt `[USER]`, `[BOT]`) akan dipecah secara berantakan oleh Tokenizer menjadi sub-kata yang kehilangan maknanya (OOV). 
Kita bisa mendaftarkan entitas baru ini agar dikenal sebagai **1 token utuh**.

> **ATURAN MUTLAK:** Jika Anda menambah ukuran *vocabulary* pada tokenizer, Anda **WAJIB** meresize embedding pada Neural Network (`model.resize_token_embeddings()`) agar sistem mengalokasikan parameter matriks dimensi teks yang baru!

In [ ]:

print(len(tokenizer))
tokenizer.add_tokens(['huawei'])
print(len(tokenizer))
token = tokenizer.tokenize(kalimat)
print(f"id : {token}")

30522
30523
id : ['huawei', "'", 's', 'large', 'models', 'are', 'very', 'fast', '!']


In [40]:
special_token_dict = {'additional_special_tokens': ['[HUAWEI AI]','[COMPETITION]']}
tokenizer.add_special_tokens(special_token_dict)
tokenizer.all_special_tokens


['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]', '[HUAWEI AI]', '[COMPETITION]']

In [ ]:
from transformers import GPT2Tokenizer

gpt_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token 
gpt_tokenizer.pad_token_id = gpt_tokenizer.eos_token_id
gpt_tokenizer.pad_token
# 4. KODE WAJIB SETELAH PENAMBAHAN TOKEN (Jalankan setelah model diinisialisasi)
# Di sel ini model belum didefinisikan secara resmi, namun kodenya adalah:
# model.resize_token_embeddings(len(tokenizer))
# print("Ukuran Vocab Model bersinergi menjadi:", len(tokenizer)) 

'<|endoftext|>'

### 8. Membuat Custom Pipeline PyTorch Dataset
Di ekosistem PyTorch, agar data berukuran raksasa bisa dimuat ke memori perlahan (per-batch), kita WAJIB mewariskan class `torch.utils.data.Dataset` dan mendapuk 3 fungsi utama (`__init__`, `__len__`, `__getitem__`). Ini adalah kompetensi wajib di kompetisi tingkat global.

In [86]:
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len, labels):
        super().__init__()
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, index):
        text = str(self.texts[index])
        label = self.labels[index]

        encoding = tokenizer(
            text,
            max_length = self.max_len,
            return_tensors='pt',
            add_special_tokens=True,
            padding='max_length'
        )
        return {
            'input_ids' : encoding['input_ids'].flatten(),
            'attention_mask' : encoding['attention_mask'].flatten(),
            'labels' : torch.tensor(label, dtype=torch.long)
        }

dummy_texts = ["Great phone", "Bad battery", "Awesome camera"]
dummy_labels = [1, 0, 1]

dataset = TextDataset(dummy_texts, tokenizer, 12, dummy_labels)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

for data in dataloader:
    print(data)

{'input_ids': tensor([[ 101, 2919, 6046,  102,    0,    0,    0,    0,    0,    0,    0,    0]]), 'attention_mask': tensor([[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]]), 'labels': tensor([0])}
{'input_ids': tensor([[  101, 12476,  4950,   102,     0,     0,     0,     0,     0,     0,
             0,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]]), 'labels': tensor([1])}
{'input_ids': tensor([[ 101, 2307, 3042,  102,    0,    0,    0,    0,    0,    0,    0,    0]]), 'attention_mask': tensor([[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]]), 'labels': tensor([1])}


### 9. Memuat Model Pre-Trained & Arsitektur Head
Selain tokenizer, kita perlu memuat arsitektur *Neural Network*-nya. `BertForSequenceClassification` artinya kita mengimpor *base* BERT, namun di kepalanya (*head*) sudah dipasangi layer Linear (Linear Layer classifier) untuk menebak kategori.
Tentukan `num_labels` sesuai jumlah kategori target.

In [ ]:
import transformers
transformers.utils.logging.set_verbosity_info()
import os

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=5)

loading configuration file config.json from cache at C:\Users\Dindin\.cache\huggingface\hub\models--bert-base-uncased\snapshots\86b5e0934494bd15c9632b12f734a8a67f723594\config.json
Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
model.to('cuda')
next(model.parameters()).device

device(type='cuda', index=0)

### 10. Ekstraksi Fitur Dasar (Hidden States & Pooler Output)
Seringkali kita tidak ingin prediksi langsung (seperti klasifikasi kelas), melainkan ingin mengambil inti vektor (representasi matematis) kalimat tersebut (dikenal sebagai kalimat *Embeddings/Features*). Ini melibatkan layer inti BERT.

In [16]:
# from transformers import BertModel
from transformers import BertModel

model = BertModel.from_pretrained('bert-base-uncased').to('cuda')
# # Panggil BertModel biasa (tanpa Classification Head)
# fitur_model = BertModel.from_pretrained(bert_path).to(device)
# fitur_model.eval()

# dummy_text = tokenizer("Learn PyTorch", return_tensors='pt').to(device)

# with torch.no_grad():
#     # Menghasilkan dua elemen penting:
#     # 1. last_hidden_state: Represenasi seluruh token kalimat individu
#     # 2. pooler_output: Representasi dari token awal [CLS] (digunakan mencakup arti keseluruhan kalimat)
#     fitur_out = fitur_model(**dummy_text)
    
#     print("Shape Last Hidden State (Batch x Seq_len x Hidden_Dim):", fitur_out.last_hidden_state.shape)
#     print("Shape Pooler Output ([CLS] token representasi, dimensi 768):", fitur_out.pooler_output.shape)

loading configuration file config.json from cache at C:\Users\Dindin\.cache\huggingface\hub\models--bert-base-uncased\snapshots\86b5e0934494bd15c9632b12f734a8a67f723594\config.json
Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.4.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading weights file model.safetensors from cache at C:\Users\Din

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
model.eval()

dummy_text = tokenizer("Learn Pytorch", return_tensors='pt').to('cuda')
print(dummy_text['input_ids'])
with torch.no_grad():
    pred = model(**dummy_text)
    print(pred['last_hidden_state'].shape)
    print(pred['pooler_output'].shape)

tensor([[  101,  4553,  1052, 22123,  2953,  2818,   102]], device='cuda:0')
torch.Size([1, 7, 768])
torch.Size([1, 768])


In [46]:
for i in model.embeddings.parameters():
    print(i.numel())    

23440896
393216
1536
768
768


### 11. Menghitung Parameter Model
Dalam skenario komputasi, Anda sering ditanya _"berapa banyak memori dan bobot angka (parameters) yang dimiliki model ini?"_ Fungsi iterasi standard `numel()` sering dipakai di kompetisi untuk melacak ini.

In [52]:
def print_model_params(model):
    total_params = sum([p.numel() for p in model.parameters()])
    trainable_params = sum([p.numel() for p in model.parameters() if p.requires_grad])
    print("total param : ", total_params)
    print("trainable param : ", trainable_params)

print_model_params(model)

total param :  109482240
trainable param :  109482240


### 12. Membekukan Layer Model (Layer Freezing)
Fine-tuning seluruh bobot model _(full fine-tuning)_ sangat boros komputasi. Teknik jitu di NLP adalah membekukan bobot awal (Embeddings & sebagian Encoder Layer awal), lalu **hanya melatih sisa kepalanya saja**.

In [53]:
model

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
for p in model.embeddings.parameters():
    p.requires_grad = False

for i in range(6):
    for p in model.encoder.layer[i].parameters():
        p.requires_grad = False

print_model_params(model=model)

total param :  109482240
trainable param :  590592


### 13. Siklus Autograd PyTorch (Backpropagation)
Ini adalah anatomi "Nyawa" pelatihan neural network. Jika ada yang terlewat, model tidak akan belajar atau justru akan rusak logikanya. Siklus rutinnya: **Zero_Grad -> Forward -> Loss -> Backward -> Step**.

In [68]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2).to('cuda')
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

loading configuration file config.json from cache at C:\Users\Dindin\.cache\huggingface\hub\models--bert-base-uncased\snapshots\86b5e0934494bd15c9632b12f734a8a67f723594\config.json
Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.4.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading weights file model.safetensors from cache at C:\Users\Din

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [83]:
dummy_input = torch.randint(0, 1000, (1, 10)).to('cuda')
dummy_label = torch.tensor([1]).to('cuda')

model.train()
optimizer.zero_grad()
outputs = model(input_ids=dummy_input, labels=dummy_label)
loss, logits = outputs.loss, outputs.logits
loss.backward()
optimizer.step()

### 14. Mode Evaluasi (Inference), Forward Pass, & Perhitungan Loss
Tahap inferensi dan forward dari model di Pytorch melibatkan beberapa tahap presisi. Pastikan tensor dikirim ke `device` yang sama dengan letak Model, lalu matikan kalkulasi gradient (menggunakan `torch.no_grad()`) apabila sedang tidak di mode *Training* untuk menghemat RAM.

In [104]:
model.eval()
sample_batch = next(iter(dataloader))
input_ids = sample_batch['input_ids'].to('cuda')
attention_mask = sample_batch['attention_mask'].to('cuda')
labels = sample_batch['labels'].to('cuda')

with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    print(outputs.loss)
    print(outputs.logits)
    print(outputs.logits.argmax(dim=1))

tensor(0.8123, device='cuda:0')
tensor([[-0.1891,  0.0366]], device='cuda:0')
tensor([1], device='cuda:0')


### 15. Menyimpan & Memuat Model Checkpoint (Save & Load `.pt`)
Proses final setelah training berjam-jam adalah mengekstrak matriks bobot (*State Dictionary*) ke hard-drive lokal. Kita juga bisa menyimpan riwayat optimizernya.

In [106]:
state_dict_to_save = {
    'model_state_dict' : model.state_dict()
}
torch.save(state_dict_to_save, 'model_bert.pt')

In [108]:
checkpoint = torch.load('model_bert.pt')
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

### 16. Mixed Precision Training (AMP) - Senjata untuk VRAM Terbatas
LLM/Transformers asli membutuhkan memori (VRAM) GPU yang masif. Memanfaatkan *Automatic Mixed Precision* (AMP) PyTorch (`torch.cuda.amp.autocast`) akan membuat komputasi angka desimal diturunkan sementera (dari Float32 ke Float16/BFloat16). Ini mempercepat proses hingga 2x dan memangkas pemakaian memori 50%!

In [ ]:
from torch.cuda.amp import autocast, GradScaler

# Scaler digunakan mencegah underflow (angka terlalu kecil nyaris lenyap) saat operasi FP16
scaler = GradScaler()

# Contoh bayangan di dalam looping iterasi epoch:
# optimizer.zero_grad()
# 
# with autocast():  <--- Kunci Utamanya
#     outputs = model(input_ids, attention_mask=mask, labels=labels)
#     loss = outputs.loss
#
# scaler.scale(loss).backward()
# scaler.step(optimizer)
# scaler.update()
print("Trik AMP autocast() sangat disarankan diterapkan di pipeline Large Model!")

### 17. Gradient Accumulation (Eksekusi Batch Global Besar di RAM Kecil)
Large model sering butuh batch_size = 32 untuk menstabilkan algoritma optimasi, tetapi GPU kita mungkin hanya muat batch_size = 4 (OOM Error). Solusinya: Kumpulkan gradient sejumlah siklus, lalu update sekalian!

In [ ]:
accumulation_steps = 4 # Artinya kita tumpuk 4 siklus batch

# Looping semu iterasi data:
for i, batch_data in enumerate([1, 2, 3, 4, 5, 6, 7, 8]):
    
    # 1. Hitung Loss biasa 
    # outputs = model(...)
    # loss = outputs.loss
    
    # 2. Nilai loss dinormalkan berdasarkan berapa lapis akumulasi
    # loss = loss / accumulation_steps
    # loss.backward()
    
    # 3. Baru update bobot optimizer dan membuang riwayat jika kelipatan step tercapai (4, 8, dst).
    if (i + 1) % accumulation_steps == 0:
        # optimizer.step()
        # optimizer.zero_grad()
        print(f"Iterasi ke-{i+1}: Bobot Optimizer Diupdate (Sistem seolah menerima batch size besar)")